# GPU Budget Negotiation Arena - Colab Training Notebook

This notebook is the judge-facing training path for the environment. It keeps the workflow small enough for hackathon compute while still producing reproducible reward traces, baseline comparisons, and a clear before/after story.


## 1. Setup

Clone the repository, install minimal dependencies, and run the local test suite before training.


In [ ]:
%cd /content
!git clone https://github.com/abhinavgautam01/GPU_Budget_Negotiation_Arena.git
%cd GPU_Budget_Negotiation_Arena
!python -m pip install -q -e ".[dev]"
!python -m pytest -q


## 2. Generate warm-start traces

These traces are for action formatting and negotiation shape, not for final evaluation.


In [ ]:
!python scripts/generate_sft_data.py --seeds 25 --output data/sft_traces.jsonl
!python scripts/evaluate_baselines.py --seeds 10 --output artifacts/baseline_eval.json
!python scripts/plot_eval.py --input artifacts/baseline_eval.json --output plots/baseline_rewards.png


## 3. Optional SFT warm start with Unsloth

Use a small instruct model and only teach valid JSON actions plus basic negotiation patterns. Keep this stage short.


In [ ]:
# Example only. Uncomment when ready.
# !pip install -q unsloth trl transformers datasets accelerate peft
# MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
# SFT_OUTPUT = "artifacts/sft-checkpoint"


## 4. GRPO training skeleton

This cell shows the structure expected by judges. Replace the placeholders with your chosen model and reward wiring once you have the final training environment.


In [ ]:
from gpu_budget_arena.env import GpuBudgetNegotiationEnv
from gpu_budget_arena.models import ResetConfig

env = GpuBudgetNegotiationEnv()
obs = env.reset(ResetConfig(task_type="market_round", seed=11))
print(obs.task_id, obs.difficulty, len(obs.private_jobs))

# Pseudocode for final GRPO setup:
# from trl import GRPOConfig, GRPOTrainer
# trainer = GRPOTrainer(
#     model=model,
#     processing_class=tokenizer,
#     reward_funcs=[format_reward, env_reward],
#     args=GRPOConfig(output_dir="artifacts/grpo-run", max_steps=300),
#     train_dataset=train_dataset,
# )
# trainer.train()


## 5. Before / after evaluation

Use the same evaluation seeds for baseline and trained policy. Save the results into `artifacts/` and commit the plot images into `plots/` for the final submission.


In [ ]:
!python scripts/evaluate_baselines.py --seeds 10 --output artifacts/baseline_eval.json
!python scripts/plot_eval.py --input artifacts/baseline_eval.json --output plots/baseline_rewards.png
